In [ ]:
"""
Kaggle Submission Generation - Robust Parquet Writer
=====================================================
Generates submission.parquet file for NFL Big Data Bowl 2026 - Prediction competition.

CRITICAL REQUIREMENTS:
- Columns: row_id, x, y (exactly 3 columns)
- row_id format: <game_id>_<play_id>_<nfl_id>_<frame_id> (string, no decimals)
- Only rows where player_to_predict == True
- Output to /kaggle/working/submission.parquet
- Parquet must use pyarrow engine with explicit schema to prevent type coercion
"""

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os
import numpy as np

# ============================================================================
# STEP 1: Load and filter data
# ============================================================================
print("[+] Loading test_input.csv...")
input_path = "/kaggle/input/nfl-big-data-bowl-2026-prediction/test_input.csv"
if not os.path.exists(input_path):
    # Fallback for local testing
    input_path = "data/test_sample/test_input.csv"

df = pd.read_csv(input_path)
print(f"[✓] Loaded {len(df):,} total rows")

# Filter to only rows where player_to_predict == True
print("\n[+] Filtering to player_to_predict == True...")
if df['player_to_predict'].dtype == 'object':
    # Handle string "True"/"False" or "true"/"false"
    submission_df = df[df['player_to_predict'].astype(str).str.lower() == 'true'].copy()
else:
    # Handle boolean values
    submission_df = df[df['player_to_predict'] == True].copy()
print(f"[✓] Filtered to {len(submission_df):,} rows")

# ============================================================================
# STEP 2: Construct row_id with STRICT integer-to-string conversion
# ============================================================================
print("\n[+] Constructing row_id...")

# Convert to int64 FIRST to ensure no float contamination
game_id_int = submission_df['game_id'].astype('int64')
play_id_int = submission_df['play_id'].astype('int64')
nfl_id_int = submission_df['nfl_id'].astype('int64')
frame_id_int = submission_df['frame_id'].astype('int64')

# Verify no NaN or inf values
assert not game_id_int.isna().any(), "game_id contains NaN"
assert not play_id_int.isna().any(), "play_id contains NaN"
assert not nfl_id_int.isna().any(), "nfl_id contains NaN"
assert not frame_id_int.isna().any(), "frame_id contains NaN"

# Construct row_id using string concatenation (prevents any float conversion)
row_id = (
    game_id_int.astype(str) + '_' +
    play_id_int.astype(str) + '_' +
    nfl_id_int.astype(str) + '_' +
    frame_id_int.astype(str)
)

# Convert to object dtype but ensure all values are strings (more compatible with PyArrow)
# Using object dtype is safer for PyArrow compatibility than pandas StringDtype
# Convert to object dtype but ensure all values are strings (more compatible with PyArrow)
# Reset index to avoid alignment issues when creating DataFrame
row_id = row_id.reset_index(drop=True).astype(str).astype(object)

# Validate row_id format
print(f"[✓] Constructed {len(row_id):,} row_id values")
sample_row_ids = row_id.head(5).tolist()
print(f"[✓] Sample row_ids: {sample_row_ids}")

# Verify no decimal points in row_id
if any('.' in str(rid) for rid in row_id):
    raise ValueError("ERROR: row_id contains decimal points!")

# ============================================================================
# STEP 3: Prepare predictions (x, y)
# ============================================================================
print("\n[+] Preparing predictions...")

# Add dummy predictions (x=0.0, y=0.0)
# TODO: Replace with actual model predictions
x_pred = pd.Series([0.0] * len(submission_df), dtype='float64')
y_pred = pd.Series([0.0] * len(submission_df), dtype='float64')

# Validate predictions
assert not x_pred.isna().any(), "x contains NaN"
assert not y_pred.isna().any(), "y contains NaN"
assert np.isfinite(x_pred).all(), "x contains non-finite values"
assert np.isfinite(y_pred).all(), "y contains non-finite values"

# ============================================================================
# STEP 4: Create final submission DataFrame with explicit schema
# ============================================================================
print("\n[+] Creating submission DataFrame...")

submission = pd.DataFrame({
    'row_id': row_id,
    'x': x_pred,
    'y': y_pred
})

# CRITICAL: Ensure exact column order
submission = submission[['row_id', 'x', 'y']]

# Verify column count
assert len(submission.columns) == 3, f"Expected 3 columns, got {len(submission.columns)}"
assert list(submission.columns) == ['row_id', 'x', 'y'], f"Column order incorrect: {list(submission.columns)}"

# Verify row_id is string type (not object)
assert submission['row_id'].dtype == 'object', f"row_id dtype is {submission['row_id'].dtype}, expected object"
assert all(isinstance(x, str) for x in submission['row_id']), "row_id must contain only strings"
assert submission['x'].dtype == 'float64', f"x dtype is {submission['x'].dtype}, expected float64"
assert submission['y'].dtype == 'float64', f"y dtype is {submission['y'].dtype}, expected float64"

print(f"[✓] Submission DataFrame created: {len(submission):,} rows × {len(submission.columns)} columns")
print(f"[✓] Column dtypes: {submission.dtypes.to_dict()}")

# ============================================================================
# STEP 5: Define explicit PyArrow schema to prevent type coercion
# ============================================================================
print("\n[+] Defining PyArrow schema...")

# Explicit schema prevents any automatic type coercion
schema = pa.schema([
    pa.field('row_id', pa.string()),  # Explicitly string, not object
    pa.field('x', pa.float64()),
    pa.field('y', pa.float64()),
])

print(f"[✓] Schema defined: {schema}")

# ============================================================================
# STEP 6: Write parquet with explicit schema and safe settings
# ============================================================================
print("\n[+] Writing parquet file...")

output_path = "/kaggle/working/submission.parquet"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Convert to PyArrow Table with explicit schema
# The schema is applied here, so we don't pass it again to write_table
table = pa.Table.from_pandas(submission, schema=schema)

# Write with safe compression settings
pq.write_table(
    table,
    output_path,
    compression='snappy',  # Standard compression, widely compatible
    use_dictionary=False,  # Disable dictionary encoding to avoid issues
    write_statistics=False  # Disable statistics for compatibility
)

print(f"[✓] Parquet file written to: {output_path}")

# ============================================================================
# STEP 7: Validate the written file by reading it back
# ============================================================================
print("\n[+] Validating written file...")

# Read back the parquet file
validation_df = pd.read_parquet(output_path)

# Verify structure
assert len(validation_df) == len(submission), "Row count mismatch after write/read"
assert list(validation_df.columns) == ['row_id', 'x', 'y'], "Column mismatch after write/read"

# Verify row_id is still string and has no decimals
assert validation_df['row_id'].dtype == 'object', \
    f"row_id dtype after read: {validation_df['row_id'].dtype}"
assert all(isinstance(x, str) for x in validation_df['row_id']), "row_id must contain only strings after read"

# Check for decimal points in row_id
sample_validation_ids = validation_df['row_id'].head(10).tolist()
if any('.' in str(rid) for rid in validation_df['row_id']):
    raise ValueError("ERROR: row_id contains decimal points after write/read!")

# Verify x, y are numeric
assert pd.api.types.is_numeric_dtype(validation_df['x']), "x is not numeric after read"
assert pd.api.types.is_numeric_dtype(validation_df['y']), "y is not numeric after read"

# Verify no NaN or inf
assert not validation_df['x'].isna().any(), "x contains NaN after read"
assert not validation_df['y'].isna().any(), "y contains NaN after read"
assert np.isfinite(validation_df[['x', 'y']].values).all(), "x or y contains non-finite values after read"

print(f"[✓] Validation passed!")
print(f"[✓] File size: {os.path.getsize(output_path) / 1024:.2f} KB")

# ============================================================================
# STEP 8: Print diagnostic information
# ============================================================================
print("\n" + "="*70)
print("SUBMISSION SUMMARY")
print("="*70)
print(f"Output path: {output_path}")
print(f"Total rows: {len(submission):,}")
print(f"Expected rows: ~5,600-6,200")
print(f"Columns: {list(submission.columns)}")
print(f"\nColumn dtypes:")
for col, dtype in submission.dtypes.items():
    print(f"  {col}: {dtype}")

print(f"\nFirst 10 rows:")
print(submission.head(10).to_string())

print(f"\nLast 5 rows:")
print(submission.tail(5).to_string())

print(f"\nRow ID sample (first 10):")
for i, rid in enumerate(submission['row_id'].head(10)):
    print(f"  [{i}] {rid} (type: {type(rid).__name__}, has_decimal: {'.' in str(rid)})")

print(f"\n[✓] Submission file ready for Kaggle!")
print("="*70)
